# **Market Basket Analysis**  
It is a technique used to discover patterns in customer purchasing behavior by analyzing the combinations of products that are frequently bought together. The idea is simple: just like examining what items appear together in a shopping basket, MBA helps identify associations such as "customers who buy milk often also buy bread."  

It is widely used in retail and e-commerce to optimize store layouts, improve cross-selling strategies, design bundle offers, and power product recommendation systems. By understanding these hidden relationships between products, businesses can make smarter marketing decisions, increase sales, and enhance the overall customer shopping experience.


### 🛒 Apriori in plain steps:

1. **Look at what people buy often.**  
   Example: if many people buy *milk* or *bread*, keep those.

2. **Put frequent items together in pairs.**  
   Example: check how often *milk + bread* are bought together.

3. **Keep only pairs that are common enough.**

4. **Try bigger groups.**  
   Example: see if *milk + bread + butter* are often bought together.

5. **Stop when no bigger groups are frequent.**

6. **Make “if–then” rules.**  
   Example: *If someone buys milk, then they also often buy bread.*


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import itertools


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Groceries_dataset.csv to Groceries_dataset.csv


In [52]:
df = pd.read_csv('Groceries_dataset.csv')

In [53]:
df.head()

,Member_number,Date,itemDescription
0,1808,21-07-2015,tropical fruit
1,2552,05-01-2015,whole milk
2,2300,19-09-2015,pip fruit
3,1187,12-12-2015,other vegetables
4,3037,01-02-2015,whole milk


In [54]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38765 entries, 0 to 38764
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Member_number    38765 non-null  int64 
 1   Date             38765 non-null  object
 2   itemDescription  38765 non-null  object
dtypes: int64(1), object(2)
memory usage: 908.7+ KB


In [69]:
df['itemDescription'] = df['itemDescription'].astype(str).str.strip()

In [70]:
df["TransactionID"] = df["Member_number"].astype(str) + "_" + df["Date"]

In [107]:
one_hot = pd.crosstab(
    df['Member_number'].astype(str) + '_' + df['Date'].astype(str),
    df['itemDescription']
).astype(int)



In [95]:
if isinstance(one_hot.columns, pd.MultiIndex):
    one_hot.columns = one_hot.columns.get_level_values(-1)
    one_hot = one_hot.groupby(level=0, axis=1).max()
    one_hot = (one_hot > 0).astype(int)

In [108]:
transactions = one_hot.to_numpy()

In [109]:
transactions

array([[0, 0, 0, ..., 1, 1, 0],
       [0, 0, 0, ..., 1, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [110]:
items = one_hot.columns.to_numpy()

In [111]:
def get_support(transactions, itemset):
    """Calculate support of an itemset"""
    mask = np.ones(transactions.shape[0], dtype=bool)
    for idx in itemset:
        mask &= transactions[:, idx] == 1
    return mask.sum() / transactions.shape[0]

In [112]:
def apriori(transactions, min_support=0.05):
    """Apriori algorithm to find frequent itemsets"""
    n_items = transactions.shape[1]
    item_indices = list(range(n_items))

    freq_itemsets = []
    # Step 1: 1-itemsets
    L1 = []
    for i in item_indices:
        sup = get_support(transactions, [i])
        if sup >= min_support:
            L1.append(([i], sup))
    freq_itemsets.extend(L1)

    # Step 2+: Generate k-itemsets
    Lk = L1
    k = 2
    while Lk:
        prev_itemsets = [sorted(x[0]) for x in Lk]
        new_candidates = []
        for i in range(len(prev_itemsets)):
            for j in range(i+1, len(prev_itemsets)):
                l1, l2 = prev_itemsets[i], prev_itemsets[j]
                if l1[:-1] == l2[:-1]:  # join step
                    candidate = sorted(set(l1) | set(l2))
                    sup = get_support(transactions, candidate)
                    if sup >= min_support and (candidate, sup) not in new_candidates:
                        new_candidates.append((candidate, sup))
        if not new_candidates:
            break
        freq_itemsets.extend(new_candidates)
        Lk = new_candidates
        k += 1

    return freq_itemsets

In [113]:
def generate_rules(freq_itemsets, min_conf=0.2):
    """Generate association rules from frequent itemsets"""
    rules = []
    for itemset, sup in freq_itemsets:
        if len(itemset) > 1:
            for i in range(1, len(itemset)):
                for antecedent in itertools.combinations(itemset, i):
                    consequent = tuple([x for x in itemset if x not in antecedent])
                    sup_antecedent = get_support(transactions, antecedent)
                    conf = sup / sup_antecedent if sup_antecedent > 0 else 0
                    sup_consequent = get_support(transactions, consequent)
                    lift = conf / sup_consequent if sup_consequent > 0 else 0
                    if conf >= min_conf:
                        rules.append((antecedent, consequent, sup, conf, lift))
    return rules

In [138]:
freq_itemsets = apriori(transactions, min_support=0.008)

In [139]:
print("Frequent Itemsets:")
for itemset, sup in freq_itemsets:
    names = [items[i] for i in itemset]
    print(f"{names} -> support={sup:.2f}")

Frequent Itemsets:
['UHT-milk'] -> support=0.02
['baking powder'] -> support=0.01
['beef'] -> support=0.03
['berries'] -> support=0.02
['beverages'] -> support=0.02
['bottled beer'] -> support=0.04
['bottled water'] -> support=0.06
['brown bread'] -> support=0.04
['butter'] -> support=0.03
['butter milk'] -> support=0.02
['candy'] -> support=0.01
['canned beer'] -> support=0.05
['cat food'] -> support=0.01
['chewing gum'] -> support=0.01
['chicken'] -> support=0.03
['chocolate'] -> support=0.02
['citrus fruit'] -> support=0.05
['coffee'] -> support=0.03
['cream cheese'] -> support=0.02
['curd'] -> support=0.03
['dessert'] -> support=0.02
['detergent'] -> support=0.01
['dishes'] -> support=0.01
['domestic eggs'] -> support=0.04
['flour'] -> support=0.01
['frankfurter'] -> support=0.04
['frozen meals'] -> support=0.02
['frozen vegetables'] -> support=0.03
['fruit/vegetable juice'] -> support=0.03
['grapes'] -> support=0.01
['ham'] -> support=0.02
['hamburger meat'] -> support=0.02
['hard

In [140]:
rules = generate_rules(freq_itemsets, min_conf=0.05)

In [141]:
print("\nAssociation Rules:")
for antecedent, consequent, sup, conf, lift in rules:
    ant_names = [items[i] for i in antecedent]
    cons_names = [items[i] for i in consequent]
    print(f"{ant_names} -> {cons_names} (support={sup:.2f}, confidence={conf:.2f}, lift={lift:.2f})")


Association Rules:
['other vegetables'] -> ['rolls/buns'] (support=0.01, confidence=0.08, lift=0.76)
['rolls/buns'] -> ['other vegetables'] (support=0.01, confidence=0.09, lift=0.76)
['other vegetables'] -> ['soda'] (support=0.01, confidence=0.08, lift=0.81)
['soda'] -> ['other vegetables'] (support=0.01, confidence=0.10, lift=0.81)
['other vegetables'] -> ['whole milk'] (support=0.01, confidence=0.12, lift=0.79)
['whole milk'] -> ['other vegetables'] (support=0.01, confidence=0.09, lift=0.79)
['rolls/buns'] -> ['whole milk'] (support=0.01, confidence=0.12, lift=0.82)
['whole milk'] -> ['rolls/buns'] (support=0.01, confidence=0.09, lift=0.82)
['sausage'] -> ['whole milk'] (support=0.01, confidence=0.14, lift=0.94)
['whole milk'] -> ['sausage'] (support=0.01, confidence=0.06, lift=0.94)
['soda'] -> ['whole milk'] (support=0.01, confidence=0.11, lift=0.74)
['whole milk'] -> ['soda'] (support=0.01, confidence=0.07, lift=0.74)
['whole milk'] -> ['yogurt'] (support=0.01, confidence=0.07, l

In [143]:
import pandas as pd

rules_df = pd.DataFrame(rules, columns=['Antecedent', 'Consequent', 'Support', 'Confidence', 'Lift'])
rules_df['Antecedent'] = rules_df['Antecedent'].apply(lambda x: ', '.join([items[i] for i in x]))
rules_df['Consequent'] = rules_df['Consequent'].apply(lambda x: ', '.join([items[i] for i in x]))

rules_df.head(10)


,Antecedent,Consequent,Support,Confidence,Lift
0,other vegetables,rolls/buns,0.009490,0.080682,0.764077
1,rolls/buns,other vegetables,0.009490,0.089873,0.764077
2,other vegetables,soda,0.008889,0.075568,0.811138
3,soda,other vegetables,0.008889,0.095409,0.811138
4,other vegetables,whole milk,0.013901,0.118182,0.792274
5,whole milk,other vegetables,0.013901,0.093190,0.792274
6,rolls/buns,whole milk,0.012898,0.122152,0.818888
7,whole milk,rolls/buns,0.012898,0.086470,0.818888
8,sausage,whole milk,0.008287,0.140430,0.941424
9,whole milk,sausage,0.008287,0.055556,0.941424


In [144]:
rules_df.sort_values('Confidence', ascending=False).head(10)


,Antecedent,Consequent,Support,Confidence,Lift
8,sausage,whole milk,0.008287,0.140430,0.941424
13,yogurt,whole milk,0.010225,0.123586,0.828505
6,rolls/buns,whole milk,0.012898,0.122152,0.818888
4,other vegetables,whole milk,0.013901,0.118182,0.792274
10,soda,whole milk,0.010292,0.110473,0.740598
3,soda,other vegetables,0.008889,0.095409,0.811138
5,whole milk,other vegetables,0.013901,0.093190,0.792274
1,rolls/buns,other vegetables,0.009490,0.089873,0.764077
7,whole milk,rolls/buns,0.012898,0.086470,0.818888
0,other vegetables,rolls/buns,0.009490,0.080682,0.764077
